# ECS171 Final Project "The Gold" 


In [ ]:
#install all dependencies
%pip install pandas numpy matplotlib seaborn scikit-learn scipy torch

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached contourpy-1.3.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (5.1 kB)
  Using cached pillow-12.2.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9

In [2]:
#CORE
import numpy as np
import pandas as pd

#visualizations
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

#preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats  

#Model / Neural Network
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

#Reproducability
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True







In [5]:
#load dataset from kaggle
df = pd.read_csv('Gold_vs_Economic_Factors_2015_2026.csv')

#change from string to date type
df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date")

#inital look at dataset, types and missing values
print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\n--- dtypes ---\n{df.dtypes}")
print(f"\n--- missing values ---\n{df.isnull().sum()}")
assert df.index.is_monotonic_increasing, "Dates are not in chronological order!"
print("Dates confirmed chronological")


Shape: (4137, 4)
Date range: 2015-01-01 00:00:00 to 2026-04-29 00:00:00

--- dtypes ---
Gold_Price_XAU_USD     float64
US_Dollar_Index_DXY    float64
Crude_Oil_Price        float64
Inflation_Rate_Pct     float64
dtype: object

--- missing values ---
Gold_Price_XAU_USD     0
US_Dollar_Index_DXY    0
Crude_Oil_Price        0
Inflation_Rate_Pct     0
dtype: int64
Dates confirmed chronological


In [7]:
#preprocessing

#separate Features vs Target
features = ["US_Dollar_Index_DXY", "Crude_Oil_Price", "Inflation_Rate_Pct"]
target = "Gold_Price_XAU_USD"

#turn into dataframes
X = df[features].values
y = df[target].values.reshape(-1, 1)

#Use 90-10 Test-Train Split
split = int(len(df) * 0.90)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Train: {len(X_train)} rows")
print(f"Test:  {len(X_test)} rows")

#scale
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train = scaler_X.fit_transform(X_train)
X_test  = scaler_X.transform(X_test)

y_train = scaler_y.fit_transform(y_train)
y_test  = scaler_y.transform(y_test)


#Pytorch Tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test,  dtype=torch.float32)

Train: 3723 rows
Test:  414 rows
